<a href="https://colab.research.google.com/github/Aakritimangal14/APS_LAB/blob/main/Copy_of_Lab_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Travelling Salesman Problem using Dynamic Programming.
Problem Description:
Given a set of cities and distances, find the shortest possible route that visits each city once and returns to the starting city.

#Theory
TSP uses dynamic programming with bit masking. Store results of visited subsets to reduce repeated work.

## Algorithm
1. Start from source city
2. Mark visited cities using bit mask
3. Try all unvisited cities
4. Choose minimum cost path
5. Return to source city

In [ ]:
import sys

def tsp_with_path(dist):
    n = len(dist)
    INF = float('inf')

    # Handle edge cases for n < 2
    if n == 0:
        return 0, []
    if n == 1:
        # A tour for one city starts at 0, visits 0, and returns to 0
        return 0, [0, 0]

    # dp[mask][i] = minimum cost to reach city i, having visited cities in 'mask'
    dp = [[INF] * n for _ in range(1 << n)]
    parent = [[-1] * n for _ in range(1 << n)]

    # Start from city 0, only city 0 visited, cost is 0
    dp[1 << 0][0] = 0 # Mask for city 0 is 1 (binary 001)

    for mask in range(1 << n):
        for u in range(n):
            # If city u is not in the current mask, or not reachable with current mask, skip
            if not (mask & (1 << u)) or dp[mask][u] == INF:
                continue

            for v in range(n):
                # If city v is already visited (in the current mask), skip
                if mask & (1 << v):
                    continue

                new_mask = mask | (1 << v)
                new_cost = dp[mask][u] + dist[u][v]

                if new_cost < dp[new_mask][v]:
                    dp[new_mask][v] = new_cost
                    parent[new_mask][v] = u

    # Close the tour: find the minimum cost to return to city 0 from any other city i
    final_mask = (1 << n) - 1
    min_tour_cost = INF
    last_city_before_return = -1

    # Iterate through all cities (except city 0, which is handled as start/end) as potential last visited cities
    # before returning to city 0.
    for i in range(1, n): # Only consider cities 1 to n-1 as the last one before returning to 0
        if dp[final_mask][i] != INF: # Ensure city i was reachable with all cities visited
            cost_to_return = dp[final_mask][i] + dist[i][0]
            if cost_to_return < min_tour_cost:
                min_tour_cost = cost_to_return
                last_city_before_return = i

    # If no valid tour was found (e.g., disconnected graph, or n < 2 was not handled)
    if min_tour_cost == INF:
        return INF, []


    # Reconstruct path
    path = []
    mask = final_mask
    curr = last_city_before_return

    # Backtrack from last_city_before_return to city 0
    while curr != -1:
        path.append(curr)
        prev = parent[mask][curr]
        mask ^= (1 << curr) # Remove curr from mask to get the mask state for 'prev'
        curr = prev

    path.append(0)  # Add the starting city (city 0)
    path.reverse()  # Reverse to get path from start to end
    path.append(0)  # Complete the cycle by returning to the start city

    return min_tour_cost, path

# ----------- INPUT -----------

n_input = int(input("Enter number of cities: "))

# Input validation for n
if n_input < 0:
    print("Number of cities cannot be negative. Setting cost to infinity and path to empty.")
    cost, path = float('inf'), []
elif n_input == 0:
    print("No cities specified. Minimum cost: 0, Path: [].")
    cost, path = 0, []
elif n_input == 1:
    print("One city specified. Minimum cost: 0, Path: [0 -> 0].")
    cost, path = 0, [0, 0]
else:
    print("Enter distance matrix:")
    input_dist = []
    for i in range(n_input):
        while True:
            try:
                row_str = input(f"Enter row {i} (space-separated integers): ")
                row = list(map(int, row_str.split()))
                if len(row) != n_input:
                    print(f"Error: Row must contain {n_input} distances. Please re-enter row {i}.")
                else:
                    input_dist.append(row)
                    break
            except ValueError:
                print("Invalid input. Please enter space-separated integers.")
            except EOFError: # Handles cases where input stream might close unexpectedly
                print("Input ended prematurely. Please provide full matrix.")
                sys.exit(1) # Exit if input is incomplete
    cost, path = tsp_with_path(input_dist)

print("\nMinimum cost:", cost)
# Only print path if it's not empty, otherwise indicate N/A or appropriate message
print("Path:", " -> ".join(map(str, path)) if path else "N/A")


Enter number of cities: 0
No cities specified. Minimum cost: 0, Path: [].

Minimum cost: 0
Path: N/A


# Exercise
1. Print path of cities visited
2. Take graph input from user
3. Test with different number of cities